# 2 — CLEAN: raw Wikipedia token streams to a prose corpus

Turns the 20 raw shards produced by notebook 1 into `clean_corpus.pkl`, the artifact
every later notebook loads.

Natural Questions ships each page as a flat token stream with a parallel `is_html` flag,
so the markup is still present as tokens. That is what makes this step possible: the
rules below find structure **by tag**, never by regex over prose, which is exact. Models
never see the tags — they exist only to decide where to cut, so cleaning has to happen
while the raw tokens are still in hand.

**Nine rules**, applied as a union of token ranges to drop, then measured per rule:
`header`, `nav`, `back_matter`, `contents`, `markers`, `banner`, `filler`, `iso_date`,
plus tables cut from the prose. Together they remove **50.7%** of the raw prose words.

**An answer span can never be deleted.** After the mask is built, every token inside an
annotated answer span is un-masked, which makes "a rule ate an answer" structurally
impossible for these rules and for any added later.

**Answers located inside tables are excluded** from the dataset. NQ flattens tables into
the token stream and only 26% expose a usable header row, so any serialisation would
mislabel most of them. 3,771 of 20,000 questions go this way. Consequence: the metrics
here are **not** comparable to published NQ-Open numbers.

| output | |
|---|---|
| pages | 18,997 |
| questions kept | 16,229 of 20,000 |
| prose words | 139,936,115 → 68,999,385 |
| runtime | ~262 s, CPU only, ~0.9 GB RSS per shard |

Structure: imports → config → definitions (nothing executes) → orchestration.

In [ ]:
import os, sys, pickle, html, json, time
from collections import Counter

sys.path.insert(0, os.getcwd())      # notebooks/ holds nqrun.py
import nqrun

In [ ]:
# CONFIG — one shard for developing rules, all twenty to produce the corpus.
SHARDS  = [f'step1-shards/part_{i:03d}.pkl' for i in range(20)]

DROP_ORPHAN_PAGES = False  # KEEP pages whose every question was excluded: they act as
                           # distractors, which is what open-domain retrieval really
                           # faces. Dropping them (19% of pages) would make the
                           # benchmark easier than NQ-Open as published.
PREVIEW = 140              # chars of dropped text kept per rule in the audit

os.environ['NQ_HF_PUSH'] = '1'       # macOS defaults this off; cleaning runs locally

OUT    = nqrun.art('step1-clean')
AUDIT  = str(OUT / 'cleaning_audit.jsonl')
CORPUS = str(OUT / 'clean_corpus.pkl')
HF_DIR = 'step1-clean'

print(f'{len(SHARDS)} shard(s): {SHARDS[0]} ... {SHARDS[-1]}')
print(f'output -> {OUT}')

## Definitions

Nothing below executes; running these cells only defines names.

Every rule has the same shape — `f(toks, ish) -> [(start, end), ...]`, raw **token**
ranges to drop. That uniformity is what lets one function measure any rule and lets
`clean_page` union them all.

In [ ]:
# PRIMITIVES — the token-stream helpers every rule shares.

def word_windows(toks, ish, size):
    """Yield every window of `size` consecutive words as (token indices, texts)."""
    words = [i for i in range(len(toks)) if not ish[i]]
    for k in range(len(words) - size + 1):
        win = words[k:k + size]
        yield win, [toks[i] for i in win]


def find_phrase(toks, ish, phrase):
    """Token ranges where `phrase` appears as consecutive words."""
    out = []
    for win, text in word_windows(toks, ish, len(phrase)):
        if text == list(phrase):
            out.append((win[0], win[-1] + 1))
    return out


def words_in(toks, ish, rs):
    """Count non-HTML tokens inside the token ranges `rs`."""
    return sum(1 for a, b in rs for i in range(a, b) if not ish[i])


# NQ writes '2017-18' with an en dash; the token stream writes '2017 -- 18' with two
# hyphens (Penn-Treebank convention). We deliberately do NOT rewrite the text for
# this, so the COMPARISON is what has to be dash-insensitive, not the corpus.
# Comparing with whitespace normalisation alone reported 7 broken gold labels on shard 0,
# of which 6 were fine.
def same_answer(a, b):
    """Is `a` present in `b`, ignoring spacing and dash style?"""
    def norm(x):
        x = x.replace('--', '–').replace('—', '–').replace('-', '–')
        return ''.join(x.split()).lower()
    return norm(a) in norm(b)

In [ ]:
# RULES 1-4 — STRUCTURAL CUTS, each anchored on a tag and a heading word.
RULES = {}


# Everything up to and including </H1> is the browser-tab title echo, e.g.
# 'Robot - assisted surgery - wikipedia'. Positional, so the inconsistent
# 'Wikipedia' / 'wikipedia' casing is irrelevant.
def header(toks, ish):
    """Drop the title echo that precedes </H1>."""
    h1 = next((i for i in range(len(toks)) if ish[i] and toks[i] == '</H1>'), None)
    return [(0, h1 + 1)] if h1 is not None else []


NAV = ('Jump', 'to', ':', 'navigation', ',', 'search')


def nav(toks, ish):
    """Drop the 'Jump to : navigation , search' line that sits after </H1>."""
    return find_phrase(toks, ish, NAV)


# The first <H2> whose heading starts with a stop word, cut to the END of the page. This
# takes References, External links, the navboxes, 'Categories :', the interlanguage
# sidebar and the licence footer in one positional cut — 36.6% of all raw words, the
# single largest rule.
STOP = ('References', 'Notes', 'External', 'Sources', 'Bibliography', 'Further', 'See')


def back_matter(toks, ish):
    """Drop from the first stop-word <H2> to the end of the page."""
    n = len(toks)
    for i in range(n):
        if ish[i] and toks[i] == '<H2>':
            w = next((toks[j] for j in range(i + 1, n) if not ish[j]), None)
            if w in STOP:
                return [(i, n)]
    return []


# '<H2> Contents </H2> ( hide ) <Ul> 1 Foo 2 Bar ...' cut to the NEXT <H2>. Small in
# words, but it lists every heading on the page, so a chunk made of it is a bag of topic
# keywords that matches many queries and answers none.
def contents(toks, ish):
    """Drop the table-of-contents block, up to the next <H2>."""
    n = len(toks)
    for i in range(n):
        if ish[i] and toks[i] == '<H2>':
            w = next((toks[j] for j in range(i + 1, n) if not ish[j]), None)
            if w == 'Contents':
                nxt = next((j for j in range(i + 1, n) if ish[j] and toks[j] == '<H2>'), n)
                return [(i, nxt)]
    return []


RULES['header'] = header
RULES['nav'] = nav
RULES['back_matter'] = back_matter
RULES['contents'] = contents

In [ ]:
# RULES 5-8 — INLINE CUTS, plus the Penn-Treebank quote transform.

MARKS = (('(', 'edit', ')'), ('(', 'hide', ')'))


def markers(toks, ish):
    """Drop the '( edit )' and '( hide )' UI markers."""
    return sorted(r for m in MARKS for r in find_phrase(toks, ish, m))


# 'This article contains content written like an advertisement ...'. The marker text
# LOCATES the banner; the enclosing <Td> ... </Td> BOUNDS it, so there is no guessing
# where the prose resumes. Deliberately not triggered on '<Table> before first <P>' —
# infoboxes live there too and hold real answers.
MARKER = 'Learn how and when to remove this template message'.split()


def banner(toks, ish):
    """Drop a maintenance-template banner, bounded by its enclosing table cell."""
    out = []
    for a, b in find_phrase(toks, ish, MARKER):
        td_open = next((i for i in range(a, -1, -1) if ish[i] and toks[i] == '<Td>'), None)
        td_close = next((i for i in range(a, len(toks)) if ish[i] and toks[i] == '</Td>'), None)
        if td_open is None or td_close is None:
            out.append((a, b))                  # no cell around it: drop just the marker
        else:
            out.append((td_open, td_close + 1))
    return out


# Wikipedia's placeholder for a cell with no data yet. These articles were written
# mid-season, so unaired episodes carry whole rows of them. No NQ answer is ever 'TBD',
# but one IS 'TBA' — which is why answer spans are un-masked in `clean_page`.
FILLER = ('TBD', 'TBA')


def filler(toks, ish):
    """Drop 'TBD' / 'TBA' placeholder tokens."""
    return [(i, i + 1) for i in range(len(toks)) if not ish[i] and toks[i] in FILLER]


# Wikipedia's date template prints the date twice: 'October 22 , 2017 ( 2017 - 10 - 22 )'.
# The 7 tokens in parens repeat the previous 4. The window is 8 words wide so the
# spelled-out year is included: only drop the copy when the year inside the parens EQUALS
# the word just before '(', which leaves a standalone ISO date that is real content.
def iso_date(toks, ish):
    """Drop the parenthesised ISO date that duplicates the spelled-out one before it."""
    out = []
    for win, text in word_windows(toks, ish, 8):
        year, open_p, y, dash1, mm, dash2, dd, close_p = text
        if (open_p == '(' and close_p == ')' and dash1 == '-' and dash2 == '-'
                and y == year
                and len(y) == 4 and y.isdigit()
                and len(mm) == 2 and mm.isdigit()
                and len(dd) == 2 and dd.isdigit()):
            out.append((win[1], win[7] + 1))        # drop '( ... )', keep the spelled date
    return out


RULES['markers'] = markers
RULES['banner'] = banner
RULES['filler'] = filler
RULES['iso_date'] = iso_date

# REJECTED — inserting '|' cell separators from the </Td> / </Th> / </Tr> tags, so that a
# flattened table row read '12 | " The Key " | March 18 , 2018'. Pointless once table text
# is cut from the prose rather than serialised into it, and it left 7,050 stray '|' in the
# prose when it was on. Do not re-add it.

# PTB QUOTES — a TRANSFORM, not a cut. NQ writes `` The Key '' ; a reader would copy that
# spelling into its answers. Applied to page text AND to the rendered answer, so the
# training target stays verbatim in its chunk.
QMAP = {'``': '"', "''": '"'}

In [ ]:
# FEATURES — extracted from regions we delete, instead of being lost. Each takes the MASK
# as well, so a feature inherits every rule. Without it, 77% of headings carried
# '( edit )' and 22% came from deleted sections.
FEATURES = {}


def headings(toks, ish, mask):
    """Every <H2>/<H3> heading that survived cleaning, in page order."""
    out, n = [], len(toks)
    for i in range(n):
        if ish[i] and toks[i] in ('<H2>', '<H3>') and not mask[i]:
            j, w = i + 1, []
            while j < n and not (ish[j] and toks[j] in ('</H2>', '</H3>')):
                if not ish[j] and not mask[j]:
                    w.append(toks[j])
                j += 1
            if w:
                out.append(' '.join(w))
    return out


# Wikipedia's curated topic list lives in the back matter, which `back_matter` cuts — so
# this one reads the RAW tokens on purpose and ignores the mask. It takes the LAST
# 'Categories :' on the page: the first can be inside a citation, which once returned
# "Judaism and Gospel '' . Bibleinterp.com". Stops at 'Hidden', the maintenance list.
def categories(toks, ish, mask):
    """Wikipedia's category list, read from the raw tokens and ignoring the mask."""
    w = [i for i in range(len(toks)) if not ish[i]]
    hits = [k for k in range(len(w) - 1)
            if toks[w[k]] == 'Categories' and toks[w[k + 1]] == ':']
    if not hits:
        return []
    out = []
    for j in w[hits[-1] + 2:]:
        if toks[j] == 'Hidden':
            break
        out.append(toks[j])
    return out


FEATURES['headings'] = headings
FEATURES['categories'] = categories

In [ ]:
# THE MASK, THE TABLE CUT, AND CLEANING ONE PAGE.

def rule_mask(toks, ish):
    """Build the drop mask from every rule in RULES, with a per-rule audit record."""
    # `tables` is NOT in RULES: it needs this mask as its input, so it runs afterwards.
    mask, per_rule = bytearray(len(toks)), {}
    for name, f in RULES.items():
        rs = f(toks, ish)
        if rs:
            txt = ' '.join(toks[i] for a, b in rs for i in range(a, b) if not ish[i])
            per_rule[name] = {'n_ranges': len(rs), 'n_words': words_in(toks, ish, rs),
                              'preview': txt[:PREVIEW]}
        for a, b in rs:
            for i in range(a, b):
                mask[i] = 1
    return mask, per_rule


# We only need to know WHERE tables are. Their text is cut out of the prose so it cannot
# pollute prose chunks, and an answer span landing inside a cut range simply fails to
# resolve — which is how table-borne answers get excluded. Nothing is parsed: no rows, no
# cells, no colspans. The structure is NOT stored, because the raw checkpoint holds every
# token of every page, so tables can be re-parsed from it whenever table retrieval units
# are built. An earlier version that expanded colspans stored one episode synopsis seven
# times: 1,290,004 duplicated words in shard 0, 83% of all stored table text.
def table_ranges(toks, ish, mask):
    """Token range of every kept <Table>, so its text can be cut from the prose."""
    out, i, n = [], 0, len(toks)
    while i < n:
        if ish[i] and toks[i].startswith('<Table') and not mask[i]:
            end = next((j for j in range(i, n) if ish[j] and toks[j] == '</Table>'), n - 1) + 1
            out.append((i, end))
            i = end
        else:
            i += 1
    return out


def clean_page(toks, ish, spans=()):
    """Clean one page: returns (words, raw->clean index map, n_tables, audit, mask)."""
    mask, per_rule = rule_mask(toks, ish)

    # An answer is never deletable. Un-masking every answer token here makes "a rule ate
    # an answer" structurally impossible — for these rules and for any added later — so
    # the `spans lost` gate is a guarantee rather than something to watch. Found via the
    # question "when is the last movie of divergent coming out", whose answer IS the word
    # 'TBA' that the filler rule removed.
    for q in spans:
        for s_, e_ in q['raw_spans']:
            for i in range(s_, e_):
                mask[i] = 0

    rs = table_ranges(toks, ish, mask)        # found BEFORE the mask grows
    if rs:
        per_rule['tables'] = {'n_ranges': len(rs),
                              'n_words': words_in(toks, ish, rs),
                              'preview': f'{len(rs)} tables cut'}
        for a, b in rs:
            for i in range(a, b):
                mask[i] = 1

    words, imap = [], {}
    for i in range(len(toks)):
        if mask[i] or ish[i]:
            continue
        imap[i] = len(words)
        words.append(html.unescape(QMAP.get(toks[i], toks[i])))
    return words, imap, len(rs), per_rule, mask

In [ ]:
# REVIEW HELPERS — used only by the optional rule-review pass, which sets the globals
# `docs`, `by_doc` and `WRD` before calling these.

def spans_hit(h, rs):
    """Answer spans on page `h` overlapping a dropped range — must be 0 for every rule."""
    return sum(1 for q in by_doc.get(h, []) for s, e in q['raw_spans']
               if any(s < b and e > a for a, b in rs))


def measure(name):
    """Print how many pages one rule fires on, how many words it drops, and spans hit."""
    f, pages, w, hit = RULES[name], 0, 0, 0
    for h, d in docs.items():
        rs = f(d['token'], d['is_html'])
        if rs:
            pages += 1
            w += words_in(d['token'], d['is_html'], rs)
            hit += spans_hit(h, rs)
    print(f'{name:<12} fires on {pages:>4}/{len(docs)} pages | drops {w:>8,} words '
          f'({w / WRD:>5.1%}) | spans hit {hit}')

## Orchestration

Two passes over the shards. The first builds the question index across **all** shards,
because a shard carries only the pages that were new in its window — so a question in
shard 5 can point at a page first seen in shard 2. The second cleans one shard at a time
and frees it, which keeps memory flat at one shard of pages rather than the whole 1.4 GB
corpus.

In [ ]:
# PASS 1 — the question index, from EVERY shard.
QIDX, ALLQ = {}, []
for sh in SHARDS:
    ck = pickle.load(open(nqrun.hf_pull(sh), 'rb'))
    for q in ck['questions']:
        QIDX.setdefault(q['doc_hash'], []).append(q)
        ALLQ.append(q)          # STREAM order - the only place it survives, and the only
    del ck                      # thing a later val split can be derived from
print(f'{len(ALLQ):,} questions over {len(QIDX):,} distinct pages')

In [ ]:
# RULE REVIEW — OPTIONAL. Skip it to produce the corpus; run it when changing a rule.
# It re-runs every rule once per rule, so it costs ~9 extra passes over one shard.
REVIEW_SHARD = SHARDS[0]
_ck = pickle.load(open(nqrun.hf_pull(REVIEW_SHARD), 'rb'))
docs, qs = _ck['documents'], _ck['questions']
by_doc = {h: QIDX.get(h, []) for h in docs}
WRD = sum(1 for d in docs.values() for b in d['is_html'] if not b)

for name in RULES:
    measure(name)
print(f"pages with NO stop heading: "
      f"{sum(1 for d in docs.values() if not back_matter(d['token'], d['is_html']))}")

n_t = w_cut = hit = 0
for h, d in docs.items():
    toks, ish = d['token'], d['is_html']
    mask, _ = rule_mask(toks, ish)
    rs = table_ranges(toks, ish, mask)
    n_t += len(rs)
    w_cut += words_in(toks, ish, rs)
    hit += spans_hit(h, rs)
print(f'tables       {n_t:>5} cut | {w_cut:,} words ({w_cut / WRD:.0%}) | '
      f'answer spans inside one: {hit} (those questions are EXCLUDED)')
for name, f in FEATURES.items():
    vals = [f(d['token'], d['is_html'], rule_mask(d['token'], d['is_html'])[0])
            for d in docs.values()]
    print(f'{name:<12} found on {sum(1 for v in vals if v):>4}/{len(docs)} pages')
del _ck

In [ ]:
# PASS 2 — clean every shard, one at a time, and free it.
corpus, audit = {}, []
st = Counter()
t0 = time.time()
for sh in SHARDS:
    ck = pickle.load(open(nqrun.hf_pull(sh), 'rb'))
    for h, d in ck['documents'].items():
        toks, ish = d['token'], d['is_html']
        qlist = QIDX.get(h, [])
        w, imap, n_tbl, per_rule, mask = clean_page(toks, ish, spans=qlist)
        corpus[h] = {'title': d['title'], 'urls': d['urls'], 'words': w,
                     **{name: f(toks, ish, mask) for name, f in FEATURES.items()}}
        raw = sum(1 for b in ish if not b)
        audit.append({'doc_hash': h, 'title': d['title'], 'shard': sh,
                      'n_words_raw': raw, 'n_words_kept': len(w),
                      'pct_removed': round(1 - len(w) / raw, 4) if raw else 0.0,
                      'n_tables': n_tbl,
                      'removed': per_rule})
        page = ' '.join(w)
        for q in qlist:
            gold, rendered = [], []
            for s_, e_ in q['raw_spans']:
                pos = [imap[i] for i in range(s_, e_) if i in imap]
                if pos:
                    lo, hi = min(pos), max(pos) + 1
                    if len(pos) != hi - lo: st['widened'] += 1
                    gold.append(('text', lo, hi)); rendered.append(' '.join(w[lo:hi]))
                    st['in_text'] += 1
                else:
                    # DECISION: answers inside tables are NOT indexed. NQ flattens tables
                    # into a token stream and only 26% expose a usable header row, so any
                    # serialisation would mislabel most of them. The raw checkpoint keeps
                    # the structure; these questions are counted and excluded. Metrics are
                    # therefore NOT comparable to published NQ-Open numbers, nor to this
                    # project's earlier runs, both of which included table answers.
                    st['answer_in_table'] += 1
            if not gold:
                st['q_dropped'] += 1; continue
            q['gold'], q['answers_rendered'] = gold, rendered
            st['q_kept'] += 1
            st['verbatim'] += (rendered[0] in page if gold[0][0] == 'text'
                               else same_answer(q['answer'], rendered[0]))
    del ck
    print(f'  {sh}: {len(corpus):,} pages cleaned, {time.time() - t0:.0f}s elapsed')

if DROP_ORPHAN_PAGES:
    with_q = {q['doc_hash'] for q in ALLQ if 'gold' in q}
    orphans = [h for h in corpus if h not in with_q]
    for h in orphans:
        del corpus[h]
    audit = [a for a in audit if a['doc_hash'] in corpus]
    print(f'  dropped {len(orphans):,} pages left with no question')

questions = [q for q in ALLQ if 'gold' in q]      # stream order preserved
print(f'\ncleaned {len(corpus):,} pages / {len(questions):,} questions '
      f'in {time.time() - t0:.0f}s')

In [ ]:
# GATES + the audit file.
with open(AUDIT, 'w') as fh:
    for rec in audit:
        fh.write(json.dumps(rec, ensure_ascii=False) + '\n')

raw_w = sum(r['n_words_raw'] for r in audit)
kept_w = sum(len(c['words']) for c in corpus.values())
with_q = {q['doc_hash'] for q in ALLQ if 'gold' in q}
unresolved = sum(1 for h in with_q if h not in corpus)   # pages a KEPT question needs
print(f"prose words {raw_w:,} -> {kept_w:,}  ({1 - kept_w / raw_w:.1%} removed)")
print(f"tables cut from the prose: {sum(a['n_tables'] for a in audit):,} "
      f"(structure not stored - re-parse from the raw checkpoint if needed)")
print(f"audit -> {AUDIT}  ({os.path.getsize(AUDIT) / 1e6:.2f} MB)")
print(f"\nGOLD UNITS")
print(f"  answers in prose   : {st['in_text']:,}")
print(f"  answers in a table : {st['answer_in_table']:,}  "
      f"({st['answer_in_table'] / max(st['in_text'] + st['answer_in_table'], 1):.0%})"
      f"  <- EXCLUDED by decision")
print(f"\nGATES")
print(f"  spans lost entirely             : {st['lost']}        <- must be 0")
print(f"  spans widened by cleaning       : {st['widened']}        <- must be 0")
print(f"  questions dropped entirely      : {st['q_dropped']}")
print(f"  answer recoverable from its unit: {st['verbatim']:,}/{st['q_kept']:,}")
print(f"  pages referenced but never seen : {unresolved}        <- must be 0")

In [ ]:
# INSPECT — one page, and one answer in its unit. Change PAGE to step through.
PAGE, WORDS = 0, 120
h = list(corpus)[PAGE]
c = corpus[h]
print(f"=== {c['title']}   {len(c['words']):,} prose words, "
      f"{len(c['headings'])} headings")
print('  ' + ' '.join(c['words'][:WORDS]) + ' ...\n')

q = next((q for q in questions if q['doc_hash'] == h), None)
if q:
    lo, hi = q['gold'][0][1], q['gold'][0][2]
    print(f"QUESTION  {q['question']}")
    print(f"NQ ANSWER {q['answer']!r}   target {q['answers_rendered']!r}")
    print(f"context   ...{' '.join(c['words'][max(0, lo - 12):hi + 12])}...")

In [ ]:
# SAVE + push. The cleaned corpus is the artifact every later notebook loads; the raw
# checkpoint stays the lossless record, so cleaning can always be redone.
Q_KEYS = ('question', 'answer', 'answers', 'answers_rendered', 'doc_hash', 'url',
          'raw_spans', 'gold')
pickle.dump({'documents': corpus,
             'questions': [{k: q[k] for k in Q_KEYS} for q in questions],
             'rules': list(RULES), 'features': list(FEATURES),
             'shards': SHARDS,
             'dataset_revision': 'e8103d566bef4154c2c12b17c6095ec5275840cc'},
            open(CORPUS, 'wb'))
print(f'-> {CORPUS}  ({os.path.getsize(CORPUS) / 1e6:,.0f} MB)')

nqrun.hf_push(CORPUS, f'{HF_DIR}/clean_corpus.pkl',
              f'cleaned corpus: {len(corpus):,} pages, {len(questions):,} questions')
nqrun.hf_push(AUDIT, f'{HF_DIR}/cleaning_audit.jsonl', 'per-page cleaning audit')
nqrun.log_metrics('2_clean', n_pages=len(corpus), n_questions=len(questions),
                  words_raw=raw_w, words_kept=kept_w,
                  answers_in_text=st['in_text'], answers_in_table=st['answer_in_table'],
                  spans_lost=st['lost'], spans_widened=st['widened'],
                  verbatim=st['verbatim'], shards=len(SHARDS))